# Notebook 4: NULL Handling and Feature Importance

**Series:** Random Forests & Isolation Forests for Full-Stack Engineers  
**Prerequisites:** Notebooks 1–3  
**Author:** [Farty Bobo](https://fartybobo.com)
**What you'll learn:**
1. Smart strategies for handling missing values (NULLs) in your features
2. How to ask a trained forest "which features actually matter?"

---

## Part 1: Handling NULLs

### The Problem

Real-world data is messy. Sensors fail. Forms are incomplete. ETL jobs crash. You'll have NaN values. sklearn's decision trees don't handle NaN natively — they'll throw an error.

But the bigger question is: **WHY is this value missing?** That changes your strategy.

### The Engineering Analogy: Sentinel Values

You've probably used sentinel values in your code — a special out-of-band value that signals "this is different":
- `null` / `undefined` in JavaScript
- `-1` as "not found" in index searches
- `0.0.0.0` as "no IP address"

The same idea applies in ML for NULLs: you can impute (fill in) a sentinel value that says "this was missing" to the model, and the model can learn to split on it.

---

### Three Strategies Based on Why Data is Missing

#### Strategy 1: Drop the rows
Use when: NULLs are very rare (< 1%) and random. No information is lost.
```python
df.dropna(subset=['feature_x'])
```

#### Strategy 2: Impute with median/mean
Use when: missing = "wasn't recorded" with no meaningful pattern. The absent data has no signal.
```python
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
```

#### Strategy 3: Impute with out-of-range sentinel
Use when: missing IS meaningful — you want the model to learn different behavior for missing values.

The trick from Tom McT's notebook: set NULL to **10% above the max value** of the feature.

Why 10% above max? 
- It's outside the normal range, so the tree can cleanly separate NULLs from non-NULLs with one split
- But it's close enough that we don't waste too much split resolution in the big gap
- The forest can learn: "if this value is at the sentinel, treat it differently"

```python
# Compute sentinel value per feature
feature_min  = X_train[:, col].min()
feature_max  = X_train[:, col].max()
feature_range = feature_max - feature_min
sentinel = feature_max + 0.1 * feature_range  # 10% above max

# Replace NaNs with sentinel
X_train[np.isnan(X_train[:, col]), col] = sentinel
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import load_iris, make_blobs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

pal = sns.color_palette('colorblind')
sns.set_palette(pal)

iris = load_iris()
print('Setup complete.')

In [ ]:
# === Demonstrate the out-of-range sentinel strategy ===
# We'll take the Iris dataset, inject artificial NULLs at varying rates,
# and compare accuracy across imputation strategies.

def inject_nulls(X, null_rate, random_state=42):
    """Randomly set null_rate fraction of values to NaN in each column."""
    rng = np.random.RandomState(random_state)
    X_nulled = X.astype(float).copy()
    for col in range(X.shape[1]):
        n_nulls = int(X.shape[0] * null_rate)
        null_indices = rng.choice(X.shape[0], size=n_nulls, replace=False)
        X_nulled[null_indices, col] = np.nan
    return X_nulled

def impute_median(X_train, X_test):
    """Fill NaNs with the median computed from training data only."""
    imputer = SimpleImputer(strategy='median')
    return imputer.fit_transform(X_train), imputer.transform(X_test)

def impute_sentinel(X_train, X_test, margin=0.1):
    """Fill NaNs with a sentinel value 10% above the training data's max."""
    X_train_out = X_train.copy()
    X_test_out  = X_test.copy()
    for col in range(X_train.shape[1]):
        col_min = np.nanmin(X_train[:, col])
        col_max = np.nanmax(X_train[:, col])
        col_range = col_max - col_min
        sentinel = col_max + margin * col_range
        X_train_out[np.isnan(X_train_out[:, col]), col] = sentinel
        X_test_out[np.isnan(X_test_out[:, col]),   col] = sentinel
    return X_train_out, X_test_out


X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

null_rates   = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]
strategies   = ['Median Imputation', 'Sentinel Imputation']
results      = {s: [] for s in strategies}

for null_rate in null_rates:
    X_train_nulled = inject_nulls(X_train, null_rate)
    X_test_nulled  = inject_nulls(X_test,  null_rate, random_state=99)
    
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    
    # Strategy 1: median imputation
    Xtr_med, Xte_med = impute_median(X_train_nulled, X_test_nulled)
    rf.fit(Xtr_med, y_train)
    results['Median Imputation'].append(rf.score(Xte_med, y_test))
    
    # Strategy 2: sentinel imputation
    Xtr_sent, Xte_sent = impute_sentinel(X_train_nulled, X_test_nulled)
    rf.fit(Xtr_sent, y_train)
    results['Sentinel Imputation'].append(rf.score(Xte_sent, y_test))

fig, ax = plt.subplots(figsize=(9, 5), dpi=100)
for i, (strategy, scores) in enumerate(results.items()):
    ax.plot([r * 100 for r in null_rates], scores, marker='o', lw=2.5,
            label=strategy, color=pal.as_hex()[i])

ax.set_xlabel('NULL rate (% of values that are missing)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('NULL Imputation Strategy Comparison on Iris Dataset', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Note: Sentinel works better when the model can learn something FROM the missingness.')
print('      Median works better when missing data carries no signal.')

---

## Part 2: Feature Importance — What Actually Mattered?

After training, you naturally want to know: **which features drove the predictions?**

This is incredibly useful for:
- Debugging: if a feature the model thinks is important shouldn't be, something is wrong
- Data collection: where should you invest in better measurements?
- Feature selection: can you drop low-importance features to speed up your pipeline?
- Explaining predictions to stakeholders

sklearn offers two approaches:

### Approach 1: Impurity-Based Importance (`model.feature_importances_`)

For each feature, sum up how much it reduced Gini impurity across all splits in all trees, weighted by the number of samples at each split.

**Fast:** computed for free during training.  
**Weakness:** biased toward high-cardinality features (features with many unique values get more split opportunities, so they appear more important even if they aren't).

### Approach 2: Permutation Importance (`permutation_importance()`)

After training:
1. Measure baseline accuracy on the test set
2. For each feature, **randomly shuffle** that feature's values and re-measure accuracy
3. The drop in accuracy = how important that feature was

Shuffling breaks the relationship between a feature and the labels. If accuracy drops a lot, that feature was carrying a lot of signal. If accuracy barely changes, the feature was noise.

**Slower:** requires re-evaluating the model many times.  
**Better:** model-agnostic, unbiased, works on any model. Recommended.

In [ ]:
# Train a Random Forest on Iris and compute both types of feature importance

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print(f'Test accuracy: {rf.score(X_test, y_test):.1%}')

In [ ]:
# --- Approach 1: Impurity-based importance ---
# rf.feature_importances_ gives one importance score per feature
# rf.estimators_ gives access to each individual tree

impurity_importances = rf.feature_importances_
impurity_std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

sorted_idx = np.argsort(impurity_importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
x = np.arange(len(iris.feature_names))
bars = ax.bar(
    x,
    impurity_importances[sorted_idx],
    yerr=impurity_std[sorted_idx],
    color=pal.as_hex()[0],
    alpha=0.85,
    capsize=5
)
ax.set_xticks(x)
ax.set_xticklabels([iris.feature_names[i] for i in sorted_idx], rotation=20, ha='right')
ax.set_ylabel('Mean Gini Impurity Decrease', fontsize=11)
ax.set_title('Impurity-Based Feature Importance\n(error bars = std across trees)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Impurity-based ranking:')
for rank, idx in enumerate(sorted_idx):
    print(f'  {rank+1}. {iris.feature_names[idx]:25s}  {impurity_importances[idx]:.4f}')

In [ ]:
# --- Approach 2: Permutation importance ---
# This runs the model multiple times with each feature shuffled
# n_repeats=30 means: shuffle each feature 30 times and average the accuracy drop

perm_result = permutation_importance(
    rf,
    X_test, y_test,   # use TEST data, not training data!
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

perm_importances = perm_result.importances_mean  # average accuracy drop per feature
perm_std         = perm_result.importances_std   # std across the 30 shuffles

sorted_idx_perm = np.argsort(perm_importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
x = np.arange(len(iris.feature_names))
ax.bar(
    x,
    perm_importances[sorted_idx_perm],
    yerr=perm_std[sorted_idx_perm],
    color=pal.as_hex()[1],
    alpha=0.85,
    capsize=5
)
ax.set_xticks(x)
ax.set_xticklabels([iris.feature_names[i] for i in sorted_idx_perm], rotation=20, ha='right')
ax.set_ylabel('Mean Accuracy Drop When Shuffled', fontsize=11)
ax.set_title('Permutation Feature Importance (recommended)\n(error bars = std across 30 shuffles)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Permutation-based ranking:')
for rank, idx in enumerate(sorted_idx_perm):
    print(f'  {rank+1}. {iris.feature_names[idx]:25s}  drop={perm_importances[idx]:.4f} ± {perm_std[idx]:.4f}')

### What Do These Charts Tell You?

Both methods agree: **petal width and petal length are by far the most important features** for classifying iris species. This matches what we saw visually in Notebook 1.

Notice the error bars (std deviation):
- Small error bars = feature consistently has this importance level
- Large error bars = importance is noisy, treat with skepticism

Features with importance near zero or with error bars crossing zero are **not meaningfully contributing** — you could consider dropping them.

---

### When the Rankings Disagree

If impurity-based and permutation-based rankings disagree significantly:
- Trust **permutation importance** — it's less biased
- The disagreement often means a feature has high cardinality (many unique values) and the impurity-based method was fooled
- Feature with high impurity importance but low permutation importance? Probably a weak signal that got inflated

---

## Putting It Together: Feature Importance for Debugging

In [ ]:
# === Feature importance as a debugging tool ===
# Scenario: we've accidentally added a random noise feature to our dataset.
# Can feature importance detect it?

np.random.seed(42)

# Add 3 pure noise features to the iris dataset
noise_features = np.random.randn(len(iris.data), 3)  # completely random
X_with_noise   = np.hstack([iris.data, noise_features])

feature_names_extended = list(iris.feature_names) + ['random_noise_1', 'random_noise_2', 'random_noise_3']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_with_noise, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

rf_debug = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_debug.fit(X_tr, y_tr)

perm_debug = permutation_importance(rf_debug, X_te, y_te, n_repeats=30, random_state=42, n_jobs=-1)

sorted_idx_d = np.argsort(perm_debug.importances_mean)[::-1]

fig, ax = plt.subplots(figsize=(10, 4), dpi=100)
colors = [pal.as_hex()[0] if 'noise' not in feature_names_extended[i] 
          else pal.as_hex()[3] 
          for i in sorted_idx_d]

ax.bar(
    range(len(sorted_idx_d)),
    perm_debug.importances_mean[sorted_idx_d],
    yerr=perm_debug.importances_std[sorted_idx_d],
    color=colors,
    alpha=0.85,
    capsize=5
)
ax.set_xticks(range(len(sorted_idx_d)))
ax.set_xticklabels([feature_names_extended[i] for i in sorted_idx_d], rotation=20, ha='right')
ax.set_ylabel('Mean Accuracy Drop', fontsize=11)
ax.set_title('Feature Importance With Noise Features Added\n(orange = known noise)', fontsize=12)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Features with importance ≈ 0 (or negative!) are not contributing — safe to drop.')
print('Negative importance means shuffling that feature actually improved accuracy — it was hurting!')

The noise features cluster near zero, clearly distinguished from the real signal features. Feature importance caught them.

This is a powerful debugging technique for real projects: add feature importance as part of your model evaluation pipeline. If a feature that *shouldn't* matter shows high importance, it's a signal of data leakage, a data quality issue, or a spurious correlation.

---

## Summary

| Concept | Key Point |
|---|---|
| **NULL strategy depends on why data is missing** | Missing = not recorded → median. Missing = meaningful → sentinel |
| **Out-of-range sentinel** | Set NULL to 10% above max so the tree can split NULLs cleanly |
| **Impurity-based importance** | Fast, free, but biased toward high-cardinality features |
| **Permutation importance** | Slower but unbiased. Use this one. Works on any model. |
| **Importance as debugging** | Near-zero importance = feature isn't contributing. Negative = feature hurts. |

---

## What's Next

**Notebook 5: SHAP Values & Isolation Forests** — Go deeper on explainability (why did the model predict THIS for THIS specific input?), and learn about Isolation Forests for anomaly detection.